1. Read csv file using spark dataframe reader

In [0]:
account_key = dbutils.secrets.get(scope='databricks-scope' ,key = 'databricks-strg-access-key' )

In [0]:
spark.conf.set("fs.azure.account.key.databricksrg2026.dfs.core.windows.net",account_key)

In [0]:
circuits_df = spark.read.\
  option("header",True).\
    option("inferSchema",True).\
      csv("abfss://raw@databricksrg2026.dfs.core.windows.net/circuits.csv")


In [0]:
type(circuits_df)

In [0]:
circuits_df.show()

In [0]:
display(circuits_df)

2. Specify Schema

In [0]:
circuits_df.printSchema()

In [0]:
circuits_df.describe().show()

inferschema

StructType

In [0]:
from pyspark.sql.types import StructType , StructField , IntegerType , StringType , DoubleType


In [0]:
circuits_schema = StructType([
  StructField("circuitId",IntegerType(),False),
  StructField("circuitRef",StringType(),True),
  StructField("name",StringType(),True),
  StructField("location",StringType(),True),
  StructField("country",StringType(),True),
  StructField("lat",DoubleType(),
              True),
  StructField("lng",DoubleType(),
              True),
  StructField("alt",IntegerType(),
              True),
  StructField("url",StringType(),
              True) ])

In [0]:
circuits_df = spark.read.\
  option("header",True).\
    schema(circuits_schema).\
      csv("abfss://raw@databricksrg2026.dfs.core.windows.net/circuits.csv")

In [0]:
display(circuits_df)

In [0]:
circuits_df.show()

In [0]:
circuits_df.printSchema()


3. selecting only required columns 

Type1 : selcting only specific columns

In [0]:
circuits_selected_df = circuits_df.select("circuitId","circuitRef","name","location","country","lat","lng","alt")
display(circuits_selected_df)

Type 2: use df name.column name

In [0]:
circuits_selected_df  =  circuits_df.select(circuits_df.circuitId,circuits_df.circuitRef,circuits_df.name,circuits_df.location,circuits_df.country,circuits_df.lat,circuits_df.lng,circuits_df.alt)
display(circuits_selected_df)


Type 3  - df name["colname"]

In [0]:
circuits_selected_df =circuits_df.select(["circuitId","circuitRef","name","location","country","lat","lng","alt"])
display(circuits_selected_df)

type 4 - using func

In [0]:
from pyspark.sql.functions import col

In [0]:
circuits_selected_df = circuits_df.select(col("circuitId"),col("circuitRef"),col("name"),col("location"),col("country"),col("lat"),col("lng"),col("alt"))
display(circuits_selected_df)


we can rename using alias when using a col func

In [0]:
circuits_selected_df = circuits_df.select(col("circuitId"),col("circuitRef"),col("name"),col("location").alias("race_location"),col("country"),col("lat"),col("lng"),col("alt"))
display(circuits_selected_df)

rename

In [0]:
circuits_rename_df = circuits_df.withColumnRenamed("circuitId","circuit_id").withColumnRenamed("circuitRef","circuit_ref").withColumnRenamed("lat","latitude").withColumnRenamed("lng","longitude").withColumnRenamed("alt","altitude")
display(circuits_rename_df)


add new column

In [0]:
from pyspark.sql.functions import current_timestamp

In [0]:
circuits_final_df  =  circuits_rename_df.withColumn("ingestion_date",current_timestamp())


In [0]:
display(circuits_final_df)


In [0]:
from pyspark.sql.functions import lit

In [0]:
circuits_final_df  =  circuits_rename_df.withColumn("ingestion_date",current_timestamp())\
    .withColumn("env",lit("Production"))


In [0]:
display(circuits_final_df)

In [0]:
circuits_final_df  =  circuits_rename_df.withColumn("ingestion_date",current_timestamp())

In [0]:
display(circuits_final_df)

5. Write in fs as parquet

In [0]:
account_key = dbutils.secrets.get(scope='databricks-scope' ,key = 'databricks-strg-access-key' )

In [0]:
spark.conf.set("fs.azure.account.key.databricksrg2026.dfs.core.windows.net",account_key)

In [0]:

circuits_final_df.write.mode("overwrite").parquet("abfss://processed@databricksrg2026.dfs.core.windows.net/circuits")
